# Causal Alignment — GRPO Training

**Model:** Qwen3-14B + LoRA (r=16)  
**Algorithm:** GRPO (Group Relative Policy Optimization)  
**Train data:** `output/train.jsonl` — 102,050 rows (CLadder synthetic + CauSciBench synthetic)  
**Test data:** `output/test.jsonl` — 10,426 rows (original benchmarks)

Run cells top to bottom. Training is resumable — interrupt anytime; restart with `--resume`.

In [ ]:
import json
import sys
import random
from pathlib import Path
from collections import Counter

ROOT = Path(".").resolve()
sys.path.insert(0, str(ROOT))
print("Root:", ROOT)

## 0. Build data

Runs the full data pipeline in order. Each script skips work that is already done (checkpoint files).

| Script | Output | What it does |
|---|---|---|
| `src/data/build_dataset.py` | `dataset/unified.jsonl` | CLadder HF + CLadder synthetic + CauSciBench existing + CauSciBench synthetic |
| `src/data/split_dataset.py` | `dataset/train.jsonl`, `dataset/test.jsonl` | Synthetic → train, original benchmarks → test |
| `src/data/preprocess.py` | `output/train.jsonl`, `output/test.jsonl` | Rebuild prompts, normalize labels, add metadata |

**Skip this section** if `output/train.jsonl` and `output/test.jsonl` already exist.

In [ ]:
import subprocess

# Check what already exists
to_check = {
    "dataset/unified.jsonl": ROOT / "dataset" / "unified.jsonl",
    "dataset/train.jsonl":   ROOT / "dataset" / "train.jsonl",
    "dataset/test.jsonl":    ROOT / "dataset" / "test.jsonl",
    "output/train.jsonl":    ROOT / "output"  / "train.jsonl",
    "output/test.jsonl":     ROOT / "output"  / "test.jsonl",
}
for label, path in to_check.items():
    status = f"{sum(1 for _ in open(path)):,} rows" if path.exists() else "MISSING"
    print(f"  {'✓' if path.exists() else '✗'}  {label:<30}  {status}")

In [ ]:
UNIFIED = ROOT / "dataset" / "unified.jsonl"

if UNIFIED.exists():
    print(f"unified.jsonl exists ({sum(1 for _ in open(UNIFIED)):,} rows) — skipping build_dataset.py")
else:
    print("Running build_dataset.py  (this takes a while — CLadder synthetic ~101k rows, CauSciBench uses OpenAI API)")
    result = subprocess.run(
        [sys.executable, str(ROOT / "src" / "data" / "build_dataset.py")],
        cwd=str(ROOT),
    )
    print(f"\nExit code: {result.returncode}")

In [ ]:
TRAIN_DS = ROOT / "dataset" / "train.jsonl"
TEST_DS  = ROOT / "dataset" / "test.jsonl"

if TRAIN_DS.exists() and TEST_DS.exists():
    print(f"dataset/train.jsonl ({sum(1 for _ in open(TRAIN_DS)):,} rows) and dataset/test.jsonl ({sum(1 for _ in open(TEST_DS)):,} rows) exist — skipping split_dataset.py")
else:
    print("Running split_dataset.py ...")
    result = subprocess.run(
        [sys.executable, str(ROOT / "src" / "data" / "split_dataset.py")],
        cwd=str(ROOT),
    )
    print(f"\nExit code: {result.returncode}")

In [ ]:
OUTPUT_TRAIN = ROOT / "output" / "train.jsonl"
OUTPUT_TEST  = ROOT / "output" / "test.jsonl"

if OUTPUT_TRAIN.exists() and OUTPUT_TEST.exists():
    print(f"output/train.jsonl ({sum(1 for _ in open(OUTPUT_TRAIN)):,} rows) and output/test.jsonl ({sum(1 for _ in open(OUTPUT_TEST)):,} rows) exist — skipping preprocess.py")
else:
    print("Running preprocess.py ...")
    result = subprocess.run(
        [sys.executable, str(ROOT / "src" / "data" / "preprocess.py")],
        cwd=str(ROOT),
    )
    print(f"\nExit code: {result.returncode}")

In [ ]:
# Final status — all five files should show ✓ before proceeding
print("Data status:")
for label, path in to_check.items():
    status = f"{sum(1 for _ in open(path)):,} rows" if path.exists() else "MISSING"
    print(f"  {'✓' if path.exists() else '✗'}  {label:<30}  {status}")

## 1. Inspect training data

In [ ]:
with open(ROOT / "output" / "train.jsonl") as f:
    train_rows = [json.loads(l) for l in f]

with open(ROOT / "output" / "test.jsonl") as f:
    test_rows = [json.loads(l) for l in f]

print(f"Train: {len(train_rows):,}  |  Test: {len(test_rows):,}")
print()
print("Train sources:", dict(Counter(r["source"] for r in train_rows)))
print("Test  sources:", dict(Counter(r["source"] for r in test_rows)))

In [ ]:
# CLadder sample
cl = next(r for r in train_rows if r["source"] == "cladder")
print("=== CLadder sample ===")
print("id:", cl["id"])
print("label:", cl["label"])
print("query type:", cl["groundtruth"]["step2"])
print()
print("--- prompt (first 800 chars) ---")
print(cl["prompt"][:800])

In [ ]:
# CauSciBench sample
cs = next(r for r in train_rows if r["source"] == "causcibench")
print("=== CauSciBench sample ===")
print("id:", cs["id"])
print("label:", cs["label"])
print("method:", cs["groundtruth"]["step2"])
print("step1 gt:", cs["groundtruth"]["step1"])
print()
print("--- prompt (first 800 chars) ---")
print(cs["prompt"][:800])

In [ ]:
# Label distributions
cl_train = [r for r in train_rows if r["source"] == "cladder"]
print("CLadder train labels:", dict(Counter(r["label"] for r in cl_train)))

cl_query_types = Counter(r["groundtruth"]["step2"] for r in cl_train)
print("CLadder query types:", dict(cl_query_types.most_common()))

cs_train = [r for r in train_rows if r["source"] == "causcibench"]
print("\nCauSciBench methods:", dict(Counter(r["groundtruth"]["step2"] for r in cs_train)))

## 2. Sanity-check rewards

Verify the reward functions work on synthetic completions before loading the full model.

In [ ]:
from src.training.reward import compute_rewards

row = random.choice(cl_train)
print("Prompt query type:", row["groundtruth"]["step2"])
print("Label:", row["label"])

gt_qt = row["groundtruth"]["step2"]
gt_ans = row["label"]

good = f"""## Step 1: Causal Structure
X -> V1, V1 -> Y

## Step 2: Query Classification
{gt_qt}

## Step 3: Derive Estimand
E[Y|do(X=1)] - E[Y|do(X=0)] using do-calculus adjustment.

## Step 4: Compute
```python
p = 0.7 - 0.4
print(f"result={p}")
```

## Step 5: Answer
Answer: {gt_ans.capitalize()}"""

bad = """## Step 1: Causal Structure
no arrows

## Step 2: Query Classification
marginal

## Step 5: Answer
Answer: No"""

rewards = compute_rewards([good, bad], [row, row])
print(f"\nGood completion reward: {rewards[0]}")
print(f"Bad  completion reward: {rewards[1]}")

In [ ]:
# CauSciBench reward check
cs_row = random.choice(cs_train)
gt_method = cs_row["groundtruth"]["step2"]
gt_val = cs_row["groundtruth"]["step5"]
print("Method:", gt_method, " | GT value:", gt_val)

cs_good = f"""## Step 1: Causal Structure
- treatment: treatment_var
- outcome: outcome_var
- controls: [control1, control2]
- instrument: none

## Step 2: Method Selection
{gt_method}

## Step 3: Estimation Specification
Standard estimation using {gt_method}.

## Step 4: Implement
```python
result = {gt_val}
print(f"result={{result}}")
```

## Step 5: Answer
Answer: {gt_val}"""

cs_rewards = compute_rewards([cs_good], [cs_row])
print("CauSciBench good reward:", cs_rewards[0])

## 3. Load model

Qwen3-14B in bfloat16 with LoRA (r=16). Reference logprobs are computed by disabling adapters on the same model — no second model copy needed (~28 GB vs ~56 GB).

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name}  {props.total_memory / 1e9:.1f} GB")

In [ ]:
from src.training.train import load_policy

MODEL_NAME = "Qwen/Qwen3-14B"
# MODEL_NAME = str(ROOT / "output" / "checkpoints" / "step_500")  # resume

model, tokenizer = load_policy(MODEL_NAME)
device = str(next(model.parameters()).device)
print("\nModel device:", device)

In [ ]:
# Memory check after loading
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        alloc = torch.cuda.memory_allocated(i) / 1e9
        reserved = torch.cuda.memory_reserved(i) / 1e9
        print(f"GPU {i}: {alloc:.1f} GB allocated  /  {reserved:.1f} GB reserved")

## 4. Generation smoke test

One forward pass to make sure the model generates coherent step-by-step completions before training.

In [ ]:
from src.training.train import generate_rollouts

sample_row = cl_train[0]
print("Prompt (first 400 chars):")
print(sample_row["prompt"][:400])
print("\nGenerating 2 completions...")

completions = generate_rollouts(model, tokenizer, sample_row["prompt"], n=2, device=device)

for i, c in enumerate(completions):
    print(f"\n{'='*60}")
    print(f"Completion {i+1}:")
    print(c[:600])

In [ ]:
# Score those completions
rewards = compute_rewards(completions, [sample_row] * 2)
print("Rewards:", rewards)
print("GT query type:", sample_row["groundtruth"]["step2"])
print("GT answer:", sample_row["label"])

## 5. Training

Calls `train()` from `src/training/train.py` directly.

**Interrupt at any time** — checkpoints are saved every `SAVE_EVERY` steps and after each epoch.  
**Resume** by setting `RESUME_FROM` to the checkpoint path.

Default hyperparameters:
| Param | Value |
|---|---|
| N rollouts | 8 |
| LoRA r | 16 |
| β (KL) | 0.01 |
| LR | 2e-5 |
| Grad accum | 8 prompts |
| Epochs | 3 |

In [ ]:
import argparse
from src.training.train import (
    BETA, GRAD_ACCUM, LOG_EVERY, MAX_EPOCHS,
    N_ROLLOUTS, OUTPUT_DIR, SANDBOX_WORKERS, SAVE_EVERY,
    LR,
)

RESUME_FROM = None  # e.g. str(ROOT / "output" / "checkpoints" / "step_500")

args = argparse.Namespace(
    model       = MODEL_NAME,
    resume      = RESUME_FROM,
    output_dir  = str(ROOT / "output" / "checkpoints"),
    epochs      = MAX_EPOCHS,
    n_rollouts  = N_ROLLOUTS,
    beta        = BETA,
    lr          = LR,
    grad_accum  = GRAD_ACCUM,
    save_every  = SAVE_EVERY,
    log_every   = LOG_EVERY,
    sandbox_workers = SANDBOX_WORKERS,
)
print("Training config:")
for k, v in vars(args).items():
    print(f"  {k}: {v}")

In [ ]:
# The model is already loaded above — pass it directly to avoid reloading.
# This replicates train() but reuses the model from cell 3.

import random
import torch
import torch.nn.functional as F
from src.training.train import generate_rollouts, sequence_logprob, grpo_loss, MAX_PROMPT_LEN, MAX_NEW_TOKENS
from src.training.reward import compute_rewards

out_dir = Path(args.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=args.lr,
    weight_decay=0.01,
)

global_step = 0

for epoch in range(args.epochs):
    random.shuffle(train_rows)
    optimizer.zero_grad()
    accum_loss = accum_reward = 0.0
    n_accum = 0

    for row in train_rows:
        # 1. Generate rollouts
        model.eval()
        completions = generate_rollouts(model, tokenizer, row["prompt"], args.n_rollouts, device)
        model.train()

        # 2. Score
        rewards_list = compute_rewards(completions, [row] * args.n_rollouts, args.sandbox_workers)
        rewards = torch.tensor(rewards_list, dtype=torch.float32, device=device)

        if rewards.std() < 1e-6:
            continue

        # 3. Tokenize
        prompt_ids = tokenizer(
            row["prompt"], return_tensors="pt", truncation=True,
            max_length=MAX_PROMPT_LEN, add_special_tokens=True,
        ).input_ids[0].to(device)

        policy_lps, ref_lps = [], []
        for completion in completions:
            comp_ids = tokenizer(
                completion, return_tensors="pt", add_special_tokens=False,
                truncation=True, max_length=MAX_NEW_TOKENS,
            ).input_ids[0].to(device)

            if comp_ids.shape[0] == 0:
                z = torch.zeros(1, device=device)
                policy_lps.append(z.squeeze().requires_grad_(True))
                ref_lps.append(z.squeeze())
                continue

            lp = sequence_logprob(model, prompt_ids, comp_ids)
            policy_lps.append(lp)

            model.disable_adapter_layers()
            with torch.no_grad():
                ref_lp = sequence_logprob(model, prompt_ids, comp_ids)
            model.enable_adapter_layers()
            ref_lps.append(ref_lp)

        # 4. GRPO loss
        loss = grpo_loss(
            torch.stack(policy_lps),
            torch.stack(ref_lps).detach(),
            rewards,
            beta=args.beta,
        )
        (loss / args.grad_accum).backward()

        accum_loss   += loss.item()
        accum_reward += rewards.mean().item()
        n_accum      += 1
        global_step  += 1

        # 5. Optimizer step
        if global_step % args.grad_accum == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

        # Log
        if global_step % args.log_every == 0 and n_accum > 0:
            print(
                f"epoch={epoch+1}  step={global_step:>6}  "
                f"loss={accum_loss/n_accum:.4f}  "
                f"reward={accum_reward/n_accum:.3f}",
                flush=True,
            )
            accum_loss = accum_reward = 0.0
            n_accum = 0

        # Checkpoint
        if global_step % args.save_every == 0:
            ckpt = out_dir / f"step_{global_step}"
            model.save_pretrained(ckpt)
            tokenizer.save_pretrained(ckpt)
            print(f"Saved → {ckpt}")

    ckpt = out_dir / f"epoch_{epoch+1}"
    model.save_pretrained(ckpt)
    tokenizer.save_pretrained(ckpt)
    print(f"Epoch {epoch+1} complete → {ckpt}")

final = out_dir / "final"
model.save_pretrained(final)
tokenizer.save_pretrained(final)
print(f"Training complete → {final}")

## 6. Post-training eval

Run the baseline eval pipeline on the test set using the trained checkpoint.

In [ ]:
# Run from terminal (eval loads its own model copy to avoid state contamination):
#
#   python src/eval/eval.py --model output/checkpoints/final --output-dir output/eval_post_grpo
#
# Or inline — reuse the already-loaded model:

from src.eval.eval import run_eval
from src.eval.metrics import aggregate_metrics
import json

# Limit to first 200 rows for a quick check; remove limit for full eval
EVAL_LIMIT = 200

model.eval()
results = run_eval(
    test_rows[:EVAL_LIMIT],
    model,
    tokenizer,
    use_llm_judge=False,   # set True if you have OPENAI_API_KEY set
    sandbox_workers=8,
)

metrics = aggregate_metrics(results)

if "cladder" in metrics:
    cl = metrics["cladder"]
    print(f"CLadder (n={cl['n']})")
    print(f"  Accuracy: {cl['accuracy']:.1f}%   Avg score: {cl['avg_score']:.1f}/100")

if "causcibench" in metrics:
    cs = metrics["causcibench"]
    print(f"CauSciBench (n={cs['n']})")
    print(f"  Method acc: {cs['method_accuracy']:.1f}%   Code rate: {cs['code_execution_rate']:.1f}%   Avg score: {cs['avg_score']:.1f}/100")